# Solution to p18

The detector produces one objectness logit, two class logits, and four raw box channels at every `4x4` cell. Only the box channels are sigmoid-transformed for target comparison. Objectness is averaged over all cells; class and box losses are averaged over the one positive cell per image. The split and fingerprint checks happen at each loss seam before the forward pass.

In [ ]:
import importlib.util
import math
import random
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED=20260901
ATOL=1e-5
RTOL=1e-5
TRAIN_IDS=('detection-train-00','detection-train-01','detection-train-02','detection-train-03')
HELDOUT_IDS=('detection-heldout-00','detection-heldout-01')

fixture_path=Path("units/B2-021-cross-modal-transformers-vision/data/vision_fixture.py")
fixture_spec=importlib.util.spec_from_file_location("p18_vision_fixture",fixture_path)
vision_fixture=importlib.util.module_from_spec(fixture_spec)
sys.modules[fixture_spec.name]=vision_fixture
fixture_spec.loader.exec_module(vision_fixture)

def seed_everything():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    torch.set_num_threads(1); torch.use_deterministic_algorithms(True)

def _to_torch(fields):
    return {name:torch.from_numpy(value.copy()) for name,value in fields.items()}

def _to_numpy(fields):
    converted={}
    for name,value in fields.items():
        if not isinstance(value,torch.Tensor): raise TypeError("batch fields must be tensors")
        converted[name]=value.detach().cpu().contiguous().numpy().copy()
    return converted

def build_train_batch(ids):
    if tuple(ids)!=TRAIN_IDS: raise ValueError("train IDs must equal immutable stored order")
    features,targets=vision_fixture.build_task_batch('detection', tuple(ids))
    return _to_torch(features),_to_torch(targets)

def build_heldout_batch(ids):
    if tuple(ids)!=HELDOUT_IDS: raise ValueError("held-out IDs must equal immutable stored order")
    features,targets=vision_fixture.build_task_batch('detection', tuple(ids))
    return _to_torch(features),_to_torch(targets)

def validate_actual(features,targets):
    role,ids=vision_fixture.validate_actual_batch('detection',_to_numpy(features),_to_numpy(targets))
    images,boxes,labels=features.get("image"),targets.get("boxes"),targets.get("labels")
    if images is None or boxes is None or labels is None: raise ValueError("missing fields")
    if images.ndim!=4 or images.shape[1:]!=(1,8,8): raise ValueError("images must be (B,1,8,8)")
    if boxes.shape!=(images.shape[0],1,4) or labels.shape!=(images.shape[0],1): raise ValueError("target shape mismatch")
    if images.dtype!=torch.float32 or boxes.dtype!=torch.float32 or labels.dtype!=torch.int64: raise ValueError("dtype mismatch")
    return role,ids

class TinyGridDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone=nn.Conv2d(1,4,3,stride=2,padding=1)
        self.objectness_head=nn.Conv2d(4,1,1)
        self.class_head=nn.Conv2d(4,2,1)
        self.box_head=nn.Conv2d(4,4,1)
    def forward(self,images):
        features=F.relu(self.backbone(images))
        return self.objectness_head(features)[:,0],self.class_head(features),self.box_head(features)

def encode_targets(boxes,labels,S=4,H=8,W=8):
    B=boxes.shape[0]
    objectness=torch.zeros((B,S,S),dtype=torch.float32)
    classes=torch.full((B,S,S),-1,dtype=torch.int64)
    encoded=torch.zeros((B,4,S,S),dtype=torch.float32)
    for b in range(B):
        x0,y0,x1,y1=(float(v) for v in boxes[b,0])
        cx,cy=(x0+x1)/2,(y0+y1)/2
        sx,sy=cx*S/W,cy*S/H
        j,i=math.floor(sx),math.floor(sy)
        objectness[b,i,j]=1
        classes[b,i,j]=labels[b,0]
        encoded[b,:,i,j]=torch.tensor([sx-j,sy-i,(x1-x0)/W,(y1-y0)/H])
    return objectness,classes,encoded

def _loss_components(model,features,targets):
    objectness_logits,class_logits,box_logits=model(features["image"])
    B=features["image"].shape[0]
    if objectness_logits.shape!=(B,4,4) or class_logits.shape!=(B,2,4,4) or box_logits.shape!=(B,4,4,4):
        raise ValueError("detector output shape mismatch")
    objectness_target,class_target,box_target=encode_targets(targets["boxes"],targets["labels"])
    positive=objectness_target.bool()
    bce=F.binary_cross_entropy_with_logits(objectness_logits,objectness_target,reduction="mean")
    ce=F.cross_entropy(class_logits.permute(0,2,3,1)[positive],class_target[positive],reduction="mean")
    predicted_boxes=torch.sigmoid(box_logits.permute(0,2,3,1))[positive]
    target_boxes=box_target.permute(0,2,3,1)[positive]
    smooth=F.smooth_l1_loss(predicted_boxes,target_boxes,reduction="mean")
    return bce+ce+2*smooth,bce,ce,smooth

def compute_loss(model,features,targets):
    validate_actual(features,targets)
    total,bce,ce,smooth=_loss_components(model,features,targets)
    compute_loss.last_components=(float(bce.detach()),float(ce.detach()),float(smooth.detach()))
    return total

def train_grid_detector(model,batch,optimizer):
    features,targets=batch
    role,ids=validate_actual(features,targets)
    if role!="train" or ids!=TRAIN_IDS: raise ValueError("optimizer batch must be the train split")
    trace=[]; model.train()
    for update in range(1,17):
        optimizer.zero_grad(set_to_none=True)
        loss=compute_loss(model,features,targets)
        components=compute_loss.last_components
        loss.backward(); optimizer.step()
        trace.append({"update":update,"loss":float(loss.detach()),"bce_obj":components[0],"ce_cls":components[1],"smooth_l1_box":components[2]})
    return tuple(trace)

def decoded_best_cell_mean_iou(model,features,targets):
    objectness_logits,_,box_logits=model(features["image"])
    encoded=torch.sigmoid(box_logits)
    ious=[]
    for b in range(features["image"].shape[0]):
        best=int(objectness_logits[b].reshape(-1).argmax())
        i,j=divmod(best,4)
        tx,ty,w_norm,h_norm=(float(v) for v in encoded[b,:,i,j])
        cx=(j+tx)*2; cy=(i+ty)*2; width=w_norm*8; height=h_norm*8
        pred=torch.tensor([cx-width/2,cy-height/2,cx+width/2,cy+height/2])
        truth=targets["boxes"][b,0]
        left_top=torch.maximum(pred[:2],truth[:2]); right_bottom=torch.minimum(pred[2:],truth[2:])
        inter=(right_bottom-left_top).clamp_min(0).prod()
        union=(pred[2:]-pred[:2]).prod()+(truth[2:]-truth[:2]).prod()-inter
        ious.append(inter/union)
    return torch.stack(ious).mean()

train_batch=build_train_batch(TRAIN_IDS)
heldout_batch=build_heldout_batch(HELDOUT_IDS)
seed_everything()
model=TinyGridDetector()
optimizer=torch.optim.AdamW(model.parameters(),lr=.05,betas=(.9,.999),eps=1e-8,weight_decay=0,amsgrad=False,foreach=False,fused=False)
model.eval()
with torch.no_grad(): heldout_loss_before=float(compute_loss(model,*heldout_batch))
training_trace=train_grid_detector(model,train_batch,optimizer)
model.eval()
with torch.no_grad():
    heldout_loss_after=float(compute_loss(model,*heldout_batch))
    heldout_mean_iou=float(decoded_best_cell_mean_iou(model,*heldout_batch))
    output_shapes=tuple(tuple(x.shape) for x in model(heldout_batch[0]["image"]))
certificate={"updates":len(training_trace),"heldout_improvement":heldout_loss_before-heldout_loss_after,"heldout_mean_iou":heldout_mean_iou,"output_shapes":output_shapes}


### Answer check

In [ ]:
assert len(training_trace)==16
assert tuple(row["update"] for row in training_trace)==tuple(range(1,17))
assert all(set(row)=={"update","loss","bce_obj","ce_cls","smooth_l1_box"} for row in training_trace)
assert certificate["output_shapes"]==((2,4,4),(2,2,4,4),(2,4,4,4))
assert certificate["heldout_improvement"]>=.02
assert certificate["heldout_mean_iou"]>=.30
assert set(TRAIN_IDS).isdisjoint(HELDOUT_IDS)
for bad in (TRAIN_IDS[::-1],TRAIN_IDS[:-1],(TRAIN_IDS[0],TRAIN_IDS[0],*TRAIN_IDS[2:]),(*TRAIN_IDS[:-1],'unknown-id'),HELDOUT_IDS):
    try: build_train_batch(bad)
    except ValueError: pass
    else: raise AssertionError("train wrapper accepted invalid IDs")
for bad in (HELDOUT_IDS[::-1],HELDOUT_IDS[:1],(HELDOUT_IDS[0],HELDOUT_IDS[0]),('unknown-id',HELDOUT_IDS[1]),TRAIN_IDS):
    try: build_heldout_batch(bad)
    except ValueError: pass
    else: raise AssertionError("held-out wrapper accepted invalid IDs")
assert torch.isfinite(compute_loss(model,*train_batch))
